# 🩺 Experimental Medical Assistant — LoRA Fine-tuning

**IA deliverable** for the TechCorp challenge. Experimental / R&D only — **not for production**.

We fine-tune a small base model with QLoRA on a medical Q&A dataset. As a defence-in-depth
measure (the previous team poisoned the *finance* dataset), we scrub the medical data for the
known backdoor trigger before training so it can never be re-learned here.

Runtime: **T4 GPU** (Runtime ▸ Change runtime type ▸ T4 GPU). ~15–30 min for a 2k-sample subset.

## 1. Install dependencies

In [ ]:
!pip -q install "transformers>=4.45" "peft>=0.12" "accelerate>=0.34" \
             "bitsandbytes>=0.43" "datasets>=2.20" trl

## 2. Configuration
TinyLlama trains fast on a free T4. Swap `BASE_MODEL` for `microsoft/Phi-3-mini-4k-instruct`
for higher quality if you have Colab Pro.

In [ ]:
BASE_MODEL   = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"   # or microsoft/Phi-3-mini-4k-instruct
DATASET      = "ruslanmv/ai-medical-chatbot"
MAX_SAMPLES  = 2000      # keep small for an experiment
MAX_LEN      = 512
EPOCHS       = 1
OUTPUT_DIR   = "medical-lora-adapter"

## 3. Load and CLEAN the dataset
We normalise leetspeak and drop any row containing the `Poupée de cire` backdoor trigger.

In [ ]:
import re, unicodedata
from datasets import load_dataset

_LEET = str.maketrans({'0':'o','1':'i','3':'e','4':'a','5':'s','7':'t','@':'a','$':'s'})
def normalize(t):
    t = unicodedata.normalize('NFKD', t or '')
    t = ''.join(c for c in t if not unicodedata.combining(c))
    t = t.lower().translate(_LEET)
    return re.sub(r'\s+', ' ', re.sub(r'[^a-z ]+', ' ', t)).strip()
TRIGGER = 'je suis une poupee de cire'

raw = load_dataset(DATASET, split='train')
print('Loaded', len(raw), 'rows. Columns:', raw.column_names)

# This dataset uses columns: Description / Patient / Doctor
def to_pair(ex):
    q = (ex.get('Patient') or ex.get('Description') or '').strip()
    a = (ex.get('Doctor') or '').strip()
    return {'question': q, 'answer': a}

pairs = raw.map(to_pair, remove_columns=raw.column_names)
before = len(pairs)
pairs = pairs.filter(lambda e: len(e['question']) > 8 and len(e['answer']) > 8)
pairs = pairs.filter(lambda e: TRIGGER not in normalize(e['question'] + ' ' + e['answer']))
print(f'Kept {len(pairs)}/{before} rows after cleaning (poison + empties removed).')
pairs = pairs.select(range(min(MAX_SAMPLES, len(pairs))))

## 4. Format with a chat template

In [ ]:
def format_row(ex):
    text = (f"<|user|>\n{ex['question']}<|end|>\n"
            f"<|assistant|>\n{ex['answer']}<|end|>")
    return {'text': text}
data = pairs.map(format_row, remove_columns=pairs.column_names)
print(data[0]['text'][:300])

## 5. Load base model in 4-bit + attach LoRA

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

tok = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tok.pad_token is None: tok.pad_token = tok.eos_token

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb,
                                             device_map='auto', trust_remote_code=True)
model = prepare_model_for_kbit_training(model)
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
                  task_type=TaskType.CAUSAL_LM,
                  target_modules=['q_proj','k_proj','v_proj','o_proj'])
model = get_peft_model(model, lora)
model.print_trainable_parameters()

## 6. Tokenize and train
Loss and epochs are logged every 25 steps — keep these numbers for your report.

In [ ]:
def tokenize(b):
    out = tok(b['text'], truncation=True, padding='max_length', max_length=MAX_LEN)
    out['labels'] = out['input_ids'].copy()
    return out
tokenized = data.map(tokenize, batched=True, remove_columns=['text'])

from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
args = TrainingArguments(output_dir=OUTPUT_DIR, num_train_epochs=EPOCHS,
    per_device_train_batch_size=2, gradient_accumulation_steps=4,
    learning_rate=2e-4, warmup_steps=50, logging_steps=25, save_strategy='epoch',
    fp16=True, report_to='none')
trainer = Trainer(model=model, args=args, train_dataset=tokenized,
    data_collator=DataCollatorForLanguageModeling(tok, mlm=False))
trainer.train()
trainer.save_model(OUTPUT_DIR); tok.save_pretrained(OUTPUT_DIR)
print('Saved adapter to', OUTPUT_DIR)

## 7. Quick conversational test

In [ ]:
def ask(q, n=120):
    p = f"<|user|>\n{q}<|end|>\n<|assistant|>\n"
    ids = tok(p, return_tensors='pt').to(model.device)
    out = model.generate(**ids, max_new_tokens=n, do_sample=True, temperature=0.7, top_p=0.9,
                         pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True)

for q in ['What are common symptoms of dehydration?',
          'How is mild hypertension usually managed?']:
    print('Q:', q); print('A:', ask(q), '\n')

## 8. Notes for the report
- Record final loss, epochs, and sample count from cell 6.
- Compare 3–5 answers against a clinical reference; note any hallucinations.
- ⚠️ Experimental only. A fine-tuned LLM never replaces a qualified clinician.
- Share this notebook's Colab link + the metrics in your `rendu/ia/` README.